In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 5b: Metrik penilaian

Cerita lengkap, batas analogi, dan ujian tingkat ada di `README.md`. Hitung dulu, baru jalankan kode.

Sebuah keranjang "buah matang":

- 8 buah matang masuk keranjang
- 2 buah mentah ikut masuk
- 1 buah matang tertinggal
- 9 buah mentah benar ditolak

1. Di antara buah dalam keranjang, berapa bagian yang memang matang?
2. Di antara semua buah matang, berapa bagian yang berhasil masuk keranjang?


## Tiga istilah

Kelas di sini artinya golongan jawaban, bukan ruang belajar. Kelas positif adalah golongan yang kita cari. Pada buah, kelas positif adalah buah matang.

Tabel empat hitungan itu bernama **confusion matrix**.

Bagian keranjang yang memang matang bernama **precision**: di antara semua tebakan positif, bagian yang benar-benar positif.

Bagian buah matang yang berhasil masuk bernama **recall**: di antara semua yang benar-benar positif, bagian yang berhasil ditebak positif.

Ukuran kesalahan pada regresi adalah satu angka seberapa jauh tebakan dari jawaban. Di sini tebakannya dua golongan, jadi kita menghitung jenis salahnya.

Rumus, setelah daftar di atas:

`precision = 8 / (8 + 2)`

`recall = 8 / (8 + 1)`


In [2]:
# Pustaka adalah kumpulan fungsi siap pakai, bukan rak buku.
# Paket adalah bundel kode yang dipasang ke lingkungan proyek.
import numpy as np
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


In [3]:
# Satu ide: susun jawaban dan tebakan sesuai daftar buah.
# Delapan pasang matang-dan-masuk, dua mentah-masuk, satu matang-tertinggal, sembilan mentah-ditolak.
y_benar = np.array([1] * 8 + [0] * 2 + [1] * 1 + [0] * 9)
y_tebak = np.array([1] * 8 + [1] * 2 + [0] * 1 + [0] * 9)
print("banyak_baris", int(len(y_benar)))


banyak_baris 20


In [4]:
# Satu ide: precision dari rumus.
print("precision_contoh", 8 / (8 + 2))


precision_contoh 0.8


In [5]:
# Satu ide: recall dari rumus.
print("recall_contoh", 8 / (8 + 1))


recall_contoh 0.8888888888888888


In [6]:
# Satu ide: fungsi pustaka menghasilkan angka yang sama dengan rumus.
# pos_label=1 menetapkan kelas positif. Bukan istilah baru.
print(
    "precision_pustaka",
    float(precision_score(y_benar, y_tebak, pos_label=1)),
)
print(
    "recall_pustaka",
    float(recall_score(y_benar, y_tebak, pos_label=1)),
)
print(
    "sama",
    bool(precision_score(y_benar, y_tebak, pos_label=1) == 8 / (8 + 2))
    and bool(recall_score(y_benar, y_tebak, pos_label=1) == 8 / (8 + 1)),
)


precision_pustaka 0.8
recall_pustaka 0.8888888888888888
sama True


## Tebakan yang selalu bilang tidak hujan

95 jam tidak hujan dan 5 jam hujan. Tebakan selalu "tidak hujan". Bagian yang cocok terlihat tinggi. Recall hujan tetap nol karena tidak ada hujan yang ditemukan.

Baseline adalah cara menebak paling sederhana, di sini selalu menebak kelas yang paling banyak. Model yang baik harus mengalahkannya pada hal yang memang kita pedulikan.


In [7]:
# Satu ide: bagian yang cocok bila tebakan selalu tidak hujan.
y_jarang = np.array([0] * 95 + [1] * 5)
tebak_jarang = np.zeros(100, dtype=int)
print("cocok_jarang", float((tebak_jarang == y_jarang).mean()))


cocok_jarang 0.95


In [8]:
# Satu ide: recall hujan pada tebakan yang sama.
penemu = int(((tebak_jarang == 1) & (y_jarang == 1)).sum())
hujan = int((y_jarang == 1).sum())
print("recall_jarang", float(penemu / hujan))


recall_jarang 0.0


## Alarm jemuran

Kelas positif adalah hujan: jemuran harus diangkat. Hujan lebih jarang daripada tidak hujan. Data ini sintetis, dibuat di memori, tanpa unduhan. Bukan data asli perusahaan.

weights adalah proporsi baris yang ditugaskan ke tiap kelas. Proporsi akhir tidak persis sama bila flip_y bukan nol, karena flip_y mengundi ulang kelas sebagian baris. class_sep mengalikan ukuran hypercube; panjang sisinya 2 kali class_sep. Hypercube adalah kotak bersudut di beberapa dimensi. stratify menjaga proporsi kelas di data latih dan di data uji. Nama ini bukan istilah baru.

DecisionTreeClassifier menyusun aturan jika-maka. max_depth=3 membatasi cabang menjadi tiga kali.


In [9]:
# Satu ide: buat data hujan sintetis. Kelas 1 lebih sedikit.
# Menurut dokumentasi scikit-learn, make_classification mengundi soal klasifikasi.
# n_samples adalah banyaknya baris.
# n_features adalah banyaknya kolom semuanya: kolom informatif, kolom redundan, kolom salinan, dan sisanya kolom acak yang tidak berguna.
# n_informative adalah banyaknya kolom informatif. Tiap kelas punya kelompok sebaran normal (simpangan baku 1) di sekitar sudut hypercube berdimensi sebanyak itu.
# Hypercube adalah kotak bersudut di beberapa dimensi. Bukan istilah baru.
# Di dalam tiap kelompok, kolom informatif diundi dari sebaran normal baku (rata-rata 0, simpangan baku 1), lalu digabung secara linear, sebelum kelompok diletakkan di sudut hypercube.
# n_redundant adalah banyaknya kolom kombinasi linear acak dari kolom informatif.
# n_repeated adalah banyaknya kolom salinan, diambil acak dari kolom informatif dan redundan.
# n_clusters_per_class adalah banyaknya kelompok pada tiap kelas.
# class_sep mengalikan ukuran hypercube. Panjang sisinya 2 kali class_sep. Makin besar, kelompok makin berjauhan dan soal makin mudah.
# flip_y adalah bagian baris yang kelasnya diundi ulang secara acak.
# weights adalah proporsi baris yang ditugaskan ke tiap kelas. Proporsi akhir tidak persis sama dengan weights bila flip_y bukan nol.
# random_state membuat undian pembuatan data ini berulang selama angkanya sama.
# Nama parameter ini bukan istilah baru.
X_hujan, y_hujan = make_classification(
    n_samples=400,
    n_features=6,
    n_informative=3,
    n_redundant=0,
    n_repeated=0,
    n_clusters_per_class=1,
    weights=[0.85, 0.15],
    class_sep=1.1,
    flip_y=0.03,
    random_state=SEED,
)
print("bagian_hujan", float(y_hujan.mean()))


bagian_hujan 0.1725


In [10]:
# Satu ide: sisihkan data uji, dengan proporsi kelas yang dijaga.
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X_hujan,
    y_hujan,
    test_size=0.3,
    random_state=SEED,
    shuffle=True,
    stratify=y_hujan,
)
print("n_latih", int(len(y_latih)))
print("n_uji", int(len(y_uji)))


n_latih 280
n_uji 120


In [11]:
# Satu ide: baseline selalu menebak kelas yang paling banyak di data latih.
# np.bincount menghitung banyaknya tiap kelas. Bukan istilah baru.
kelas_banyak = int(np.bincount(y_latih).argmax())
tebakan_baseline = np.full_like(y_uji, kelas_banyak)
print("cocok_baseline", float((tebakan_baseline == y_uji).mean()))


cocok_baseline 0.825


In [12]:
# Satu ide: recall baseline.
# zero_division=0 menulis 0 bila pembagian tidak punya penyebut. Bukan istilah baru.
print(
    "recall_baseline",
    float(recall_score(y_uji, tebakan_baseline, pos_label=1, zero_division=0)),
)


recall_baseline 0.0


In [13]:
# Satu ide: skor pohon pendek pada data latih.
pohon = DecisionTreeClassifier(max_depth=3, random_state=SEED)
pohon.fit(X_latih, y_latih)
print("skor_latih_dangkal", float(pohon.score(X_latih, y_latih)))


skor_latih_dangkal 0.9464285714285714


In [14]:
# Satu ide: tebakan pohon pendek pada data uji, dan bagian yang cocok.
tebakan_uji = pohon.predict(X_uji)
print("cocok_uji", float((tebakan_uji == y_uji).mean()))


cocok_uji 0.8916666666666667


In [15]:
# Satu ide: empat kotak pada data uji.
# Pada confusion_matrix, baris adalah jawaban benar dan kolom adalah tebakan.
tn, fp, fn, tp = confusion_matrix(y_uji, tebakan_uji, labels=[0, 1]).ravel()
print("tepat_negatif", int(tn))
print("salah_positif", int(fp))
print("salah_negatif", int(fn))
print("tepat_positif", int(tp))


tepat_negatif 94
salah_positif 5
salah_negatif 8
tepat_positif 13


In [16]:
# Satu ide: precision pada data uji.
print(
    "precision_uji",
    float(precision_score(y_uji, tebakan_uji, pos_label=1, zero_division=0)),
)


precision_uji 0.7222222222222222


In [17]:
# Satu ide: recall pada data uji.
print(
    "recall_uji",
    float(recall_score(y_uji, tebakan_uji, pos_label=1, zero_division=0)),
)


recall_uji 0.6190476190476191


Baseline sering cocok, tetapi recall hujannya nol. Pohon pendek menemukan sebagian hujan. Untuk jemuran, melewatkan hujan lebih merugikan daripada alarm palsu sesekali, jadi recall dilihat lebih dulu. Precision tetap dicek supaya keranjang tidak diisi sembarangan.

Overfitting artinya hasil bagus di data latih tetapi buruk di data baru. Pohon tanpa batas boleh menghafal baris latih. Bandingkan skor data latih dan skor data uji.


In [18]:
# Satu ide: skor pohon tanpa batas pada data latih.
pohon_dalam = DecisionTreeClassifier(max_depth=None, random_state=SEED)
pohon_dalam.fit(X_latih, y_latih)
print("skor_latih_dalam", float(pohon_dalam.score(X_latih, y_latih)))


skor_latih_dalam 1.0


In [19]:
# Satu ide: skor pohon tanpa batas pada data uji.
print("skor_uji_dalam", float(pohon_dalam.score(X_uji, y_uji)))


skor_uji_dalam 0.85


## Ujian tingkat

Ujian tingkat intermediate ada di bagian terakhir `latihan.ipynb`. Kuncinya ada di `latihan-kunci.ipynb`.
